In [1]:
from parcels import FieldSet, JITParticle, ParticleSet
import parcels

import datetime as dt
from datetime import timedelta

import numpy as np

from pathlib import Path
import geopandas as gpd

import matplotlib.pyplot as plt

import xarray as xr
from glob import glob
import cartopy
import cartopy.crs as ccrs
from cartopy.crs import Geodetic
from time import time
import warnings

warnings.simplefilter('ignore')

# Set Parameters

In [2]:
# Parameters
RNG_seed = 123

# Time
year = 2024
start_month = 6
end_month = 12
start_day = 1
end_day = 1
max_age_d = 28
# timedirection
timearrow = 1
# Timestep in minutes
dt_in_minutes = 15
output_dt_in_minutes = 15


# Box traits
min_depth_m = 0
max_depth_m = 25
box_side_length_km = 5

site_counter = 11

number_particles = 100

repeated_release = True
repeatdt_d = 1

isPapermill = False

In [3]:
isPapermill = True

In [4]:
# Standard testing parameters
if not isPapermill:
    # year = 2021
    start_month = 6
    end_month = 6
    start_day = 1
    end_day = 2
    max_age_d = 10
    # timedirection
    timearrow = 1
    # Timestep in minutes
    dt_in_minutes = 15
    output_dt_in_minutes = 15

    # Box traits
    min_depth_m = 0
    max_depth_m = 25
    box_side_length_km = 5

    site_counter = 5

    number_particles = 100

    repeated_release = False
    repeatdt_d = 5

# Read Files

In [5]:
# Get Variables from Parameters
start_date = np.datetime64(f'{year}-{start_month:02d}-{start_day:02d}', 'D')
end_date = np.datetime64(f'{year}-{end_month:02d}-{end_day:02d}', 'D')
first_day_in_year = np.datetime64(f'{year}-01-01', 'D')

start_date_str = start_date.astype(str).replace('-','')
end_date_str = end_date.astype(str).replace('-','')

start_file = (start_date - first_day_in_year).astype(int) * 4-1
end_file = (end_date - first_day_in_year + 1).astype(int) * 4+1

runtime_in_days = (end_date - start_date).tolist()
dt_min = np.timedelta64(dt_in_minutes, 'm').tolist()
dt_out_min = np.timedelta64(output_dt_in_minutes, 'm').tolist()

np.random.seed(RNG_seed)
folder_path = Path('/gxfs_work/geomar/smomw597/2025_copepods/')
save_path = folder_path / f'output/Trajectories/{year}/'
print(start_file,end_file)

607 1345


In [6]:
# Read Files
folder_path_bsh_400 = Path('/gxfs_work/geomar/smomw400/bsh_operationalmodel_data/')
folder_path_bsh_122 = Path('/gxfs_work/geomar/smomw122/bsh_operationalmodel_data/')
folder_path_static_fine = folder_path_bsh_122 / 'static_file_fine'
folder_path_static_coarse = folder_path_bsh_122 / 'static_file_coarse'

sigma_file_fine = folder_path_static_fine / 'sigma_file_fine.nc'
H0_file_fine = folder_path_static_fine / 'H0_file_fine.nc'
divH0_file_fine = folder_path_static_fine / 'divH0_file_fine.nc'
lonlat_file_fine = folder_path_static_fine / 'lonlat_file_fine.nc'

c_files_fine = sorted(folder_path_bsh_400.glob(f'c_file_fine_{year}/*'))[start_file:end_file]
z_files_fine = sorted(folder_path_bsh_400.glob(f'z_file_fine_{year}/*'))[start_file:end_file]
t_files_fine = sorted(folder_path_bsh_400.glob(f't_file_fine_{year}/*'))[start_file:end_file]
divz_files_fine = sorted(folder_path_bsh_122.glob(f'divz_file_fine_{year}/*'))[start_file:end_file]

sigma_file_coarse = folder_path_static_coarse / 'sigma_file_coarse.nc'
H0_file_coarse = folder_path_static_coarse / 'H0_file_coarse.nc'
divH0_file_coarse = folder_path_static_coarse / 'divH0_file_coarse.nc'
lonlat_file_coarse = folder_path_static_coarse / 'lonlat_file_coarse.nc'

c_files_coarse = sorted(folder_path_bsh_400.glob(f'c_file_coarse_{year}/*'))[start_file:end_file]
z_files_coarse = sorted(folder_path_bsh_400.glob(f'z_file_coarse_{year}/*'))[start_file:end_file]
t_files_coarse = sorted(folder_path_bsh_400.glob(f't_file_coarse_{year}/*'))[start_file:end_file]
divz_files_coarse = sorted(folder_path_bsh_122.glob(f'divz_file_coarse_{year}/*'))[start_file:end_file]

In [7]:
# open eta and H0 files
ds_eta_fine = xr.open_dataset(z_files_fine[0])
ds_H0_fine = xr.open_dataset(H0_file_fine)

ds_eta_coarse = xr.open_dataset(z_files_coarse[0])
ds_H0_coarse = xr.open_dataset(H0_file_coarse)

# Functions

In [8]:
# https://github.com/Yichabod/natural_disaster_pred/blob/master/cropping_coordinates.py

earth_radius = 6271.0
degrees_to_radians = np.pi / 180.0
radians_to_degrees = 180.0 / np.pi


def change_in_latitude(kms):
    'Given a distance north, return the change in latitude.'
    return (kms / earth_radius) * radians_to_degrees


def change_in_longitude(latitude, kms):
    'Given a latitude and a distance west, return the change in longitude.'
    # Find the radius of a circle around the earth at given latitude.
    r = earth_radius * np.cos(np.multiply(latitude, degrees_to_radians))
    return (kms / r) * radians_to_degrees

# Samples

In [9]:
sample_points_path = folder_path / 'output/sample_points/sample_points.geojson'
sample_point = gpd.read_file(sample_points_path).to_crs(Geodetic()).loc[site_counter]
location_id = sample_point.location_id_old
location_lat = sample_point.geometry.y
location_lon = sample_point.geometry.x

In [10]:
# Make release box
lat_release_min = location_lat - change_in_latitude(box_side_length_km / 2)
lat_release_max = location_lat + change_in_latitude(box_side_length_km / 2)
lon_release_min = location_lon - change_in_longitude(location_lat, box_side_length_km / 2)
lon_release_max = location_lon + change_in_longitude(location_lat, box_side_length_km / 2)

# Make Box around release box, to ensure that depth values are existent while establishing particles
lat_box_min = location_lat - change_in_latitude(box_side_length_km * 2)
lat_box_max = location_lat + change_in_latitude(box_side_length_km * 2)
lon_box_min = location_lon - change_in_longitude(location_lat, box_side_length_km * 2)
lon_box_max = location_lon + change_in_longitude(location_lat, box_side_length_km * 2)

In [11]:
# Build masks
fine_mask = (
    (ds_eta_fine.lat >= lat_box_min)
    & (ds_eta_fine.lat <= lat_box_max)
    & (ds_eta_fine.lon >= lon_box_min)
    & (ds_eta_fine.lon <= lon_box_max)
)
coarse_mask = (
    (ds_eta_coarse.lat >= lat_box_min)
    & (ds_eta_coarse.lat <= lat_box_max)
    & (ds_eta_coarse.lon >= lon_box_min)
    & (ds_eta_coarse.lon <= lon_box_max)
)

In [12]:
# Check if area is in fine grid or not
if fine_mask.sum() == 0:
    region_mask = coarse_mask
    elev = ds_eta_coarse.elev.isel(time=0, drop=True)
    h0 = ds_H0_coarse.H0
else:
    region_mask = fine_mask
    elev = ds_eta_fine.elev.isel(time=0, drop=True)
    h0 = ds_H0_fine.H0

# only seed in water
elev_mask = ~ elev.isnull()

# avoid weird H0 < 0 locations
h0_mask = h0 > 0

In [13]:
# Check where to place the particles horizontaly and the fraction of valid cells
seed_here = (
    (elev_mask & h0_mask)
    .where(region_mask, drop=True)
    .astype(bool)
)
fraction_valid_horizontal = seed_here.mean().data
if fraction_valid_horizontal > 0:
    number_horizontal = int(number_particles / fraction_valid_horizontal * 1.25)
    seasurface_height = (
        (elev + h0)
        .where((region_mask & h0_mask), drop=True)
    )
else:
    print('Es ist ein Fehler aufgetreten')

In [14]:
# Generate lats and lons
# uniformly distributed everywhere
# uniform in lat / lon =/= uniform in m2
release_lats = np.random.uniform(lat_release_min, lat_release_max, size=number_horizontal)
release_lons = np.random.uniform(lon_release_min, lon_release_max, size=number_horizontal)

# check validity of positions
# note that this is _not_ per water volume but per sigma!
seedable = seed_here.sel(
    lon=xr.DataArray(release_lons, dims='particle'),
    lat=xr.DataArray(release_lats, dims='particle'),
    method='nearest',
).data

# remove invalide positions and cut to legth afterwards
release_lons = release_lons[seedable][:number_particles]
release_lats = release_lats[seedable][:number_particles]

In [15]:
# Get maximal depth in Area
depth_in_release_area = h0.where(region_mask, drop=True)

# Set the maximal depth of bathymetry to the given maximal depth of Particles
depth_here = depth_in_release_area.where(
    depth_in_release_area < max_depth_m, 
    max_depth_m,
)

# Calculate the Sigma for each position
sigma_here = depth_here / seasurface_height

# Check the maximal sigma value for each particle
max_sigma_here = sigma_here.sel(
    lon=xr.DataArray(release_lons, dims='particle'),
    lat=xr.DataArray(release_lats, dims='particle'),
    method='nearest',
).data

# Check the elevation for each particle
elev_here = elev.where(
        region_mask & h0_mask, 
        drop=True,
    ).sel(
        lon=xr.DataArray(release_lons, dims='particle'),
        lat=xr.DataArray(release_lats, dims='particle'),
        method='nearest',
    ).data


# generate the depth for each particle between the maximal sigma and minimal elevation
depth = np.random.uniform(elev_here, max_sigma_here)

# Parcels

## Custom Kernel

In [16]:
# Check, wether a Particle is above 25m depth and if not, move it to 24m
def FloatParticle(particle, fieldset, time):
    if particle.depth > max_depth_m:
        particle.depth = max_depth_m - 1

def DeleteErrorParticle(particle, fieldset, time):
    if particle.state >= 50:  # This captures all Errors
        particle.delete()

# Establish aging for particles
def Aging(particle, fieldset, time):
    particle.age_sec += particle.dt
    max_age_min = fieldset.max_age_d * 60 * 60 * 24
    if particle.age_sec > max_age_min:
        particle.delete()

In [17]:
def AdvectionRK4_3D_SIGMABSH(particle, fieldset, time):
    # max_depth_m = 25
    max_depth_m = fieldset.max_depth
    time0 = time
    sig0 = particle.depth
    lat0 = particle.lat
    lon0 = particle.lon

    (u1, v1) = fieldset.UV[time0, sig0, lat0, lon0]  # horizontal velocities in deg/s

    w1 = fieldset.Wz[time0, sig0, lat0, lon0]  # this is upward in m/s rel to sig0 level

    s1 = fieldset.S[time0, sig0, lat0, lon0]
    t1 = fieldset.T[time0, sig0, lat0, lon0]

    eta1 = fieldset.eta[time0, 0, lat0, lon0]  # sea level elevation
    detadlon1 = fieldset.detadlon[time0, 0, lat0, lon0]
    detadlat1 = fieldset.detadlat[time0, 0, lat0, lon0]
    h01 = fieldset.H0[0, 0, lat0, lon0]  # reference bottom depth (for eta=0)
    dh0dlon1 = fieldset.dH0dlon[0, 0, lat0, lon0]
    dh0dlat1 = fieldset.dH0dlat[0, 0, lat0, lon0]
    h1 = h01 + eta1  # total height of water column

    wsigma1 = -w1 / h1 - sig0 / h1 * (
        u1 * (detadlon1 + dh0dlon1) + v1 * (detadlat1 + dh0dlat1)
    )

    time1 = time0 + 0.5 * particle.dt
    sig1 = max(
        0.0,
        min(min(1.0, max_depth_m / (eta1 + h01)), sig0 + wsigma1 * 0.5 * particle.dt),
    )
    # sig1 = max(0.0, min(1.0, sig0 + wsigma1 * 0.5 * particle.dt))

    lat1 = lat0 + v1 * 0.5 * particle.dt
    lon1 = lon0 + u1 * 0.5 * particle.dt

    (u2, v2) = fieldset.UV[time1, sig1, lat1, lon1]

    w2 = fieldset.Wz[time1, sig1, lat1, lon1]

    eta2 = fieldset.eta[time1, 0, lat1, lon1]
    detadlon2 = fieldset.detadlon[time1, 0, lat1, lon1]
    detadlat2 = fieldset.detadlat[time1, 0, lat1, lon1]
    h02 = fieldset.H0[0, 0, lat1, lon1]
    dh0dlon2 = fieldset.dH0dlon[0, 0, lat1, lon1]
    dh0dlat2 = fieldset.dH0dlat[0, 0, lat1, lon1]
    h2 = h02 + eta2

    wsigma2 = -w2 / h2 - sig1 / h2 * (
        u2 * (detadlon2 + dh0dlon2) + v2 * (detadlat2 + dh0dlat2)
    )

    time2 = time0 + 0.5 * particle.dt
    sig2 = max(
        0.0,
        min(min(1.0, max_depth_m / (eta2 + h02)), sig0 + wsigma2 * 0.5 * particle.dt),
    )
    # sig2 = max(0.0, min(1.0, sig0 + wsigma2 * 0.5 * particle.dt))
    lat2 = lat0 + v2 * 0.5 * particle.dt
    lon2 = lon0 + u2 * 0.5 * particle.dt

    (u3, v3) = fieldset.UV[time2, sig2, lat2, lon2]

    w3 = fieldset.Wz[time2, sig2, lat2, lon2]

    eta3 = fieldset.eta[time2, 0, lat2, lon2]
    detadlon3 = fieldset.detadlon[time2, 0, lat2, lon2]
    detadlat3 = fieldset.detadlat[time2, 0, lat2, lon2]
    h03 = fieldset.H0[0, 0, lat2, lon2]
    dh0dlon3 = fieldset.dH0dlon[0, 0, lat2, lon2]
    dh0dlat3 = fieldset.dH0dlat[0, 0, lat2, lon2]
    h3 = h03 + eta3

    wsigma3 = -w3 / h3 - sig2 / h3 * (
        u3 * (detadlon3 + dh0dlon3) + v3 * (detadlat3 + dh0dlat3)
    )

    time3 = time0 + particle.dt
    sig3 = max(
        0.0, min(min(1.0, max_depth_m / (eta3 + h03)), sig0 + wsigma3 * particle.dt)
    )
    # sig3 = max(0.0, min(1.0, sig0 + wsigma3 * particle.dt))
    lat3 = lat0 + v3 * particle.dt
    lon3 = lon0 + u3 * particle.dt

    (u4, v4) = fieldset.UV[time3, sig3, lat3, lon3]

    w4 = fieldset.Wz[time3, sig3, lat3, lon3]

    eta4 = fieldset.eta[time3, 0, lat3, lon3]
    detadlon4 = fieldset.detadlon[time3, 0, lat3, lon3]
    detadlat4 = fieldset.detadlat[time3, 0, lat3, lon3]
    h04 = fieldset.H0[0, 0, lat3, lon3]
    dh0dlon4 = fieldset.dH0dlon[0, 0, lat3, lon3]
    dh0dlat4 = fieldset.dH0dlat[0, 0, lat3, lon3]
    h4 = h04 + eta4

    wsigma4 = -w4 / h4 - sig3 / h4 * (
        u4 * (detadlon4 + dh0dlon4) + v4 * (detadlat4 + dh0dlat4)
    )

    lon4 = lon0 + (u1 + 2 * u2 + 2 * u3 + u4) / 6 * particle.dt
    lat4 = lat0 + (v1 + 2 * v2 + 2 * v3 + v4) / 6 * particle.dt
    sig4 = max(
        0.0,
        min(
            min(1.0, max_depth_m / (eta4 + h04)),
            sig0 + (wsigma1 + 2 * wsigma2 + 2 * wsigma3 + wsigma4) / 6 * particle.dt,
        ),
    )
    # sig4 = max(0.0, min(1.0, sig0 + (wsigma1 + 2 * wsigma2 + 2 * wsigma3 + wsigma4) / 6 * particle.dt))

    particle_dlon += lon4 - lon0
    particle_dlat += lat4 - lat0
    particle_ddepth += sig4 - sig0

    particle.eta = eta1
    particle.h0 = h01
    particle.wz = w1
    particle.u = u1
    particle.v = v1
    particle.wsigma = wsigma1
    particle.S = s1
    particle.T = t1
    particle.d = particle.depth

In [18]:
CustomKernel = [AdvectionRK4_3D_SIGMABSH, Aging, DeleteErrorParticle]

## Particles

In [19]:
# Establish partivle variables
particle_variables = (
    'eta', 'h0', 'wz', 
    'u', 'v', 'wsigma', 
    'S', 'T', 'd',
)
SampleParticle = parcels.JITParticle.add_variables(particle_variables)
SampleParticle = SampleParticle.add_variable('age_sec', initial=0)

## Fieldset

In [20]:
# Prepare reading of variables
fieldset_variables = [
    'U', 'V', 'Wz', 
    'H0', 'dH0dlon', 'dH0dlat',
    'eta', 'detadlon', 'detadlat',
]
variable_names = [
    'uvel', 'vvel', 'wvel', 
    'H0', 'dH0dlon', 'dH0dlat',
    'elev', 'detadlon', 'detadlat'
]

dims_xy = {'lon': 'lon', 'lat': 'lat'}
dims_xy_t = {'lon': 'lon', 'lat': 'lat', 'time': 'time'}
dims_xyz_t = {'lon': 'lon', 'lat': 'lat', 'time': 'time', 'depth': 'sigma'}

dim_dicts = [
    dims_xyz_t, dims_xyz_t, dims_xyz_t,
    dims_xy, dims_xy, dims_xy,
    dims_xy_t, dims_xy_t, dims_xy_t,
]
interp_methods = [
    'cgrid_velocity', 'cgrid_velocity', 'cgrid_velocity', 
    'cgrid_tracer', 'cgrid_tracer', 'cgrid_tracer', 
    'cgrid_velocity', 'cgrid_velocity', 'cgrid_velocity',
]

variables = dict(zip(fieldset_variables, variable_names))
dimensions = dict(zip(fieldset_variables, dim_dicts))
interp_method = dict(zip(fieldset_variables, interp_methods))

### fine fieldset

In [21]:
st_dict_fine = {'lon': lonlat_file_fine, 'lat': lonlat_file_fine, 'depth': sigma_file_fine, 'data': t_files_fine}
current_dict_fine = {'lon': lonlat_file_fine, 'lat': lonlat_file_fine, 'depth': sigma_file_fine, 'data': c_files_fine}

H0_dict_fine = {'lon': lonlat_file_fine, 'lat': lonlat_file_fine, 'data': H0_file_fine}
divH0_dict_fine = {'lon': lonlat_file_fine, 'lat': lonlat_file_fine, 'data': divH0_file_fine}
divz_dict_fine = {'lon': lonlat_file_fine, 'lat': lonlat_file_fine, 'data': divz_files_fine}
eta_dict_fine = {'lon': lonlat_file_fine, 'lat': lonlat_file_fine, 'data': z_files_fine}

In [27]:
# Build fine fieldset for Temp and salinity
st_fieldset_fine = FieldSet.from_netcdf(
    filenames={'S': st_dict_fine, 'T': st_dict_fine},
    variables={'S': 'salt', 'T': 'temp'},
    dimensions={'S': dims_xyz_t, 'T': dims_xyz_t},
    interp_method={'S': 'cgrid_tracer', 'T': 'cgrid_tracer'},
    allow_time_extrapolation=False,
    gridindexingtype='nemo',
)

In [23]:
# Build fine fieldset for all the other Values
filenames_fine = {
    'V': current_dict_fine,
    'U': current_dict_fine,
    'Wz': current_dict_fine,
    'H0': H0_dict_fine,
    'dH0dlon': divH0_dict_fine,
    'dH0dlat': divH0_dict_fine,
    'eta': eta_dict_fine,
    'detadlon': divz_dict_fine,
    'detadlat': divz_dict_fine,
}

fieldset_fine = FieldSet.from_netcdf(
    filenames=filenames_fine,
    variables=variables,
    dimensions=dimensions,
    interp_method=interp_method,
    allow_time_extrapolation=False,
    gridindexingtype='nemo',
)

### coarse fieldsets

In [24]:
st_dict_coarse = {'lon': lonlat_file_coarse, 'lat': lonlat_file_coarse, 'depth': sigma_file_coarse, 'data': t_files_coarse}
current_dict_coarse = {'lon': lonlat_file_coarse, 'lat': lonlat_file_coarse, 'depth': sigma_file_coarse, 'data': c_files_coarse}
H0_dict_coarse = {'lon': lonlat_file_coarse, 'lat': lonlat_file_coarse, 'data': H0_file_coarse}
divH0_dict_coarse = {'lon': lonlat_file_coarse, 'lat': lonlat_file_coarse, 'data': divH0_file_coarse}
eta_dict_coarse = {'lon': lonlat_file_coarse, 'lat': lonlat_file_coarse, 'data': z_files_coarse}
divz_dict_coarse = {'lon': lonlat_file_coarse, 'lat': lonlat_file_coarse, 'data': divz_files_coarse}

In [25]:
# Build coarse fieldset for Temp and salinity
st_fieldset_coarse = FieldSet.from_netcdf(
    filenames={'S': st_dict_coarse, 'T': st_dict_coarse},
    variables={'S': 'salt', 'T': 'temp'},
    dimensions={'S': dims_xyz_t, 'T': dims_xyz_t},
    interp_method={'S': 'cgrid_tracer', 'T': 'cgrid_tracer'},
    allow_time_extrapolation=False,
    gridindexingtype='nemo',
)

In [26]:
# Build coarse fieldset for all the other Values
filenames_coarse = {
    'U': current_dict_coarse,
    'V': current_dict_coarse,
    'Wz': current_dict_coarse,
    'H0': H0_dict_coarse,
    'dH0dlon': divH0_dict_coarse,
    'dH0dlat': divH0_dict_coarse,
    'eta': eta_dict_coarse,
    'detadlon': divz_dict_coarse,
    'detadlat': divz_dict_coarse,
}

fieldset_coarse = FieldSet.from_netcdf(
    filenames=filenames_coarse,
    variables=variables,
    dimensions=dimensions,
    interp_method=interp_method,
    allow_time_extrapolation=False,
    gridindexingtype='nemo',
)

### Nested fieldset

In [28]:
# Build Nested field from the fine and coarse fields
U = parcels.NestedField('U', [fieldset_fine.U, fieldset_coarse.U])
V = parcels.NestedField('V', [fieldset_fine.V, fieldset_coarse.V])

nested_fieldset = FieldSet(U, V)

Wz_add = parcels.NestedField('Wz', [fieldset_fine.Wz, fieldset_coarse.Wz])
H0_add = parcels.NestedField('H0', [fieldset_fine.H0, fieldset_coarse.H0])
dH0dlon_add = parcels.NestedField('dH0dlon', [fieldset_fine.dH0dlon, fieldset_coarse.dH0dlon])
dH0dlat_add = parcels.NestedField('dH0dlat', [fieldset_fine.dH0dlat, fieldset_coarse.dH0dlat])
eta_add = parcels.NestedField('eta', [fieldset_fine.eta, fieldset_coarse.eta])
detadlon_add = parcels.NestedField('detadlon', [fieldset_fine.detadlon, fieldset_coarse.detadlon])
detadlat_add = parcels.NestedField('detadlat', [fieldset_fine.detadlat, fieldset_coarse.detadlat])
S_add = parcels.NestedField('S', [st_fieldset_fine.S, st_fieldset_coarse.S])
T_add = parcels.NestedField('T', [st_fieldset_fine.T, st_fieldset_coarse.T])

In [ ]:
# Combine the Nested Fields to a single nested field
nested_fieldset.add_field(Wz_add)
nested_fieldset.add_field(H0_add)
nested_fieldset.add_field(dH0dlon_add)
nested_fieldset.add_field(dH0dlat_add)
nested_fieldset.add_field(eta_add)
nested_fieldset.add_field(detadlon_add)
nested_fieldset.add_field(detadlat_add)
nested_fieldset.add_field(S_add)
nested_fieldset.add_field(T_add)
# Establish max depth and max age as constant in Fieldset
nested_fieldset.add_constant('max_depth', max_depth_m)
nested_fieldset.add_constant('max_age_d', max_age_d)

# Create Particles

In [ ]:
# Build Particle Set
start_time = np.datetime64(f'{year}-{start_month:02d}-{start_day:02d}T00:00:00')

pset = ParticleSet(
    fieldset=nested_fieldset,
    pclass=SampleParticle,
    lat=release_lats,
    lon=release_lons,
    depth=depth,
    time=[start_time for n in range(number_particles)],
    repeatdt=timedelta(days=repeatdt_d),
)
if not repeated_release:
    pset.repeatdt = None

In [ ]:
filename_time = f'{start_date_str}-{end_date_str}_dt{output_dt_in_minutes}min'
filename_position = f'{location_id}_d{min_depth_m}m-{max_depth_m}m'
filename = f'Nested_{filename_time}_{filename_position}_N{number_particles}_seed{RNG_seed}.zarr'
# define Output path and name
if isPapermill:
    output_filename = str('PPmill_' + filename)
else:
    output_filename = str('TEST_' + filename)

output_path = Path(save_path, output_filename)
print(f'{output_path}')

/gxfs_work/geomar/smomw597/2025_copepods/output/Trajectories/2024/PPmill_Nested_20240601-20241201_dt15min_BE01_d0m-25m_N100_seed123.zarr


In [ ]:
# Define Outputparameters
output_particle_file = pset.ParticleFile(
    name=output_path,
    outputdt=dt_out_min,
    chunks=(number_particles, int(24 * 60 / output_dt_in_minutes)),
)

# Execute

In [ ]:
# Execute Simulation
pset.execute(
    CustomKernel,
    dt=dt_min,
    runtime=runtime_in_days,
    output_file=output_particle_file,
    verbose_progress=False,
)

INFO: Output files are stored in /gxfs_work/geomar/smomw597/2025_copepods/output/Trajectories/2024/PPmill_Nested_20240601-20241201_dt15min_BE01_d0m-25m_N100_seed123.zarr.


KeyError: "No variable named 'eta'. Variables on the dataset include ['lon', 'lat', 'layer_number', 'time', 'elev', 'decoded']"